DAY-2: Slicing, Dicing and Mutating in pyspark

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
    .appName("spark_day2")\
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/03 16:43:12 WARN Utils: Your hostname, biswajits, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/03 16:43:12 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/biswa/practice/.venv/lib/python3.11/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/08/03 16:43:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


load parquet

In [2]:
parquet_path = r"../dataset/parquet_output"

df = spark.read.parquet(parquet_path)
df

DataFrame[ID: int, FirstName: string, LastName: string, Country: string, Score: int]

# Slice:

slice easy method

In [3]:
df_1 = df.select('ID', 'FirstName', 'Country')
df_1

DataFrame[ID: int, FirstName: string, Country: string]

slice using function

In [4]:
from pyspark.sql.functions import col

df_2 = df.select(
    col('ID').alias('CustomerID'),
    col('FirstName'),
    col('Country')
)

df_2

DataFrame[CustomerID: int, FirstName: string, Country: string]

# Dice

In [5]:
df_3 = df.filter(
    (col('Country') == 'USA' ) & (col('Score') > 500)
)
df_3.show()

+---+---------+--------+-------+-----+
| ID|FirstName|LastName|Country|Score|
+---+---------+--------+-------+-----+
|  2|    Kevin|   Brown|    USA|  900|
|  3|     Mary|    NULL|    USA|  750|
+---+---------+--------+-------+-----+



# Mutate

In [6]:
from pyspark.sql.functions import when

In [7]:
df_4 = df.withColumn(
    "Performance",
    when(col("Score") >= 800, "Excellent")
    .when(col("Score") >= 500, "Average")
    .otherwise("Needs Improvement")
)
df_4.show()

+---+---------+--------+-------+-----+-----------------+
| ID|FirstName|LastName|Country|Score|      Performance|
+---+---------+--------+-------+-----+-----------------+
|  1|   Jossef|Goldberg|Germany|  350|Needs Improvement|
|  2|    Kevin|   Brown|    USA|  900|        Excellent|
|  3|     Mary|    NULL|    USA|  750|          Average|
|  4|     Mark| Schwarz|Germany|  500|          Average|
|  5|     Anna|   Adams|    USA| NULL|Needs Improvement|
+---+---------+--------+-------+-----+-----------------+



small problem

"""Write a local PySpark snippet to read raw text file and use (Slice, Dice, Mutate) skills to parse it into an ultra-clean DataFrame with 4 distinct columns:
timestamp (Proper Timestamp type)
log_level (Only the words "ERROR", "INFO", or "WARN")
user_id (Integer type, replacing any null strings with 0)
message (The final text sentence)"""

In [11]:
txt_path = r"../dataset/server_log.txt"
txt = spark.read.text(txt_path)
txt.take(100)

[Row(value='"2026-08-03 14:32:01 [ERROR] User_ID:5542 - Database connection timed out"'),
 Row(value='"2026-08-03 14:35:22 [INFO] User_ID:1120 - User logged in successfully"'),
 Row(value='"2026-08-03 14:36:00 [WARN] User_ID:null - High memory utilization detected"')]

In [15]:
from pyspark.sql.functions import split, concat, lit
txt = txt.withColumn('timestamp', concat(split('value', ' ')[0], lit(' '), split('value', ' ')[1]))\
        .withColumn('log_level', split('value', ' ')[2])\
        .withColumn('user_id', split('value', ' ')[3])\
        .withColumn('message', concat(split('value', ' ')[5], lit(' '), split('value', ' ')[6], lit(' '), split('value', ' ')[7], lit(' '), split('value', ' ')[8]))
txt = txt.select('timestamp', 'log_level', "user_id", "message")
txt.show()

+--------------------+---------+------------+--------------------+
|           timestamp|log_level|     user_id|             message|
+--------------------+---------+------------+--------------------+
|"2026-08-03 14:32:01|  [ERROR]|User_ID:5542|Database connecti...|
|"2026-08-03 14:35:22|   [INFO]|User_ID:1120|User logged in su...|
|"2026-08-03 14:36:00|   [WARN]|User_ID:null|High memory utili...|
+--------------------+---------+------------+--------------------+



In [16]:
spark.stop()